In [3]:
import os
import pandas as pd
import numpy as np
import duckdb

start with looking at the head of all the smaller files but chartevents
generating the csvs for all the smaller files: d_items,icustays,patients and admissions

In [4]:
d_items_path='../data/d_items.csv.gz'
icustays_path='../data/icustays.csv.gz'
patients_path='../data/patients.csv.gz'
admissions_path='../data/admissions.csv.gz'
chartevents_path='../data/chartevents.csv.gz'
d_items=pd.read_csv(d_items_path)
icustays=pd.read_csv(icustays_path)
patients=pd.read_csv(patients_path)
admissions=pd.read_csv(admissions_path)
print("all the files have been converted to csvs")

all the files have been converted to csvs


In [5]:
d_items.head()

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,220001,Problem List,Problem List,chartevents,General,NaN,Text,NaN,NaN
1,220003,ICU Admission date,ICU Admission date,datetimeevents,ADT,NaN,Date and time,NaN,NaN
2,220045,Heart Rate,HR,chartevents,Routine Vital Signs,bpm,Numeric,NaN,NaN
3,220046,Heart rate Alarm - High,HR Alarm - High,chartevents,Alarms,bpm,Numeric,NaN,NaN
4,220047,Heart Rate Alarm - Low,HR Alarm - Low,chartevents,Alarms,bpm,Numeric,NaN,NaN


<h5>inferences: everything except item id, label and abbrev is pretty insignificant so just the first 3 is sufficient for now and check which is most linked to preferably its chartevents</h5>



In [6]:
patients.head()

,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10000032,F,52,2180,2014 - 2016,2180-09-09
1,10000048,F,23,2126,2008 - 2010,NaN
2,10000058,F,33,2168,2020 - 2022,NaN
3,10000068,F,19,2160,2008 - 2010,NaN
4,10000084,M,72,2160,2017 - 2019,2161-02-13


<h5>inferences: subject id will be the only fruitful column in this table and will link to icustays.</h5>



In [7]:
icustays.head()

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252
2,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535
3,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032
4,10001217,27703517,34592300,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-12-19 15:42:24,2157-12-20 14:27:41,0.948113


<h5>as said above stay id is linked to subj id here but intime and outtime is pretty useful from here.</h5>



In [8]:
admissions.head()

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,English,WIDOWED,WHITE,2180-05-06 19:17:00,2180-05-06 23:30:00,0
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaN,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-06-26 15:54:00,2180-06-26 21:31:00,0
2,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaN,EW EMER.,P19UTS,EMERGENCY ROOM,HOSPICE,Medicaid,English,WIDOWED,WHITE,2180-08-05 20:58:00,2180-08-06 01:44:00,0
3,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,NaN,EW EMER.,P06OTX,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-07-23 05:54:00,2180-07-23 14:00:00,0
4,10000068,25022803,2160-03-03 23:16:00,2160-03-04 06:26:00,NaN,EU OBSERVATION,P39NWO,EMERGENCY ROOM,NaN,NaN,English,SINGLE,WHITE,2160-03-03 21:55:00,2160-03-04 06:26:00,0


<h5>inferences: not completely sure of the distinction between admittime, intime etc check doc and almost everything but the first few cols is redundant</h5>



In [9]:
# for chartevents alone:
chartevents_preview=pd.read_csv('../data/chartevents.csv.gz',nrows=5)

chartevents_preview.head()

,subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valuenum,valueuom,warning
0,10000032,29079034,39553978,18704,2180-07-23 12:36:00,2180-07-23 14:45:00,226512,39.4,39.4,kg,0
1,10000032,29079034,39553978,18704,2180-07-23 12:36:00,2180-07-23 14:45:00,226707,60,60.0,Inch,0
2,10000032,29079034,39553978,18704,2180-07-23 12:36:00,2180-07-23 14:45:00,226730,152,152.0,cm,0
3,10000032,29079034,39553978,18704,2180-07-23 14:00:00,2180-07-23 14:18:00,220048,SR (Sinus Rhythm),NaN,NaN,0
4,10000032,29079034,39553978,18704,2180-07-23 14:00:00,2180-07-23 14:18:00,224642,Oral,NaN,NaN,0


<h4>most important one linking all of them, need to take batches of data based on the subject id and then cumulate all the values from the items</h4>
<h5> itemid from d_items must be subs here and grouping done by subject_id;</h5>



In [10]:
print("patients columns:",patients.columns.tolist())
print("icustays columns:",icustays.columns.tolist())
print("admissions columns:",admissions.columns.tolist())
print("d_items columns:",d_items.columns.tolist())
print("chartevents columns:",chartevents_preview.columns.tolist())

patients columns: ['subject_id', 'gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod']
icustays columns: ['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit', 'intime', 'outtime', 'los']
admissions columns: ['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'admit_provider_id', 'admission_location', 'discharge_location', 'insurance', 'language', 'marital_status', 'race', 'edregtime', 'edouttime', 'hospital_expire_flag']
d_items columns: ['itemid', 'label', 'abbreviation', 'linksto', 'category', 'unitname', 'param_type', 'lownormalvalue', 'highnormalvalue']
chartevents columns: ['subject_id', 'hadm_id', 'stay_id', 'caregiver_id', 'charttime', 'storetime', 'itemid', 'value', 'valuenum', 'valueuom', 'warning']


In [11]:
#to create a a dictionary with most of the important vitals there are
print(d_items[d_items['label'].str.contains('heart rate', case=False, na=False)])
print("-------------------------------")
print(d_items[d_items['label'].str.contains('respiratory rate', case=False, na=False)])
print("-------------------------------")
print(d_items[d_items['label'].str.contains('systolic', case=False, na=False)])
print("-------------------------------")
print(d_items[d_items['label'].str.contains('temperature', case=False, na=False)])
print("-------------------------------")
print(d_items[d_items['label'].str.contains('diastolic', case=False, na=False)])
print("-------------------------------")
print(d_items[d_items['label'].str.contains('oxygen', case=False, na=False)])


   itemid                    label     abbreviation      linksto  \
2  220045               Heart Rate               HR  chartevents   
3  220046  Heart rate Alarm - High  HR Alarm - High  chartevents   
4  220047   Heart Rate Alarm - Low   HR Alarm - Low  chartevents   

              category unitname param_type  lownormalvalue  highnormalvalue  
2  Routine Vital Signs      bpm    Numeric             NaN              NaN  
3               Alarms      bpm    Numeric             NaN              NaN  
4               Alarms      bpm    Numeric             NaN              NaN  
-------------------------------
     itemid                           label                    abbreviation  \
28   220210                Respiratory Rate                              RR   
799  224688          Respiratory Rate (Set)          Respiratory Rate (Set)   
800  224689  Respiratory Rate (spontaneous)  Respiratory Rate (spontaneous)   
801  224690        Respiratory Rate (Total)        Respiratory Rate

In [12]:
#mapping dictionary for the vitals from d_items file:
vitals_dict={"spO2":220277,"temp(C)":223762,"heartrate":220045,"ARTsys":225309,"ARTdia":225310,"ARTmean":225312,"NBPs":220179,"NBPd":220180,"NBPm":220181,"RR":224688}
vitals,ids=list(vitals_dict.keys()),list(vitals_dict.values())
vitals_rev_dict={vitals_dict[i]:i for i in vitals_dict}
print(vitals,ids)

['spO2', 'temp(C)', 'heartrate', 'ARTsys', 'ARTdia', 'ARTmean', 'NBPs', 'NBPd', 'NBPm', 'RR'] [220277, 223762, 220045, 225309, 225310, 225312, 220179, 220180, 220181, 224688]


In [13]:
con=duckdb.connect()
queryrows=f"""
SELECT subject_id, stay_id, charttime, itemid, value, valuenum
FROM read_csv_auto('{chartevents_path}')
WHERE itemid IN {tuple(ids)}
LIMIT 25
"""
result=con.execute(queryrows).fetchdf()
result

,subject_id,stay_id,charttime,itemid,value,valuenum
0,10000032,39553978,2180-07-23 14:11:00,220179,84,84.0
1,10000032,39553978,2180-07-23 14:11:00,220180,48,48.0
2,10000032,39553978,2180-07-23 14:11:00,220181,56,56.0
3,10000032,39553978,2180-07-23 14:12:00,220045,91,91.0
4,10000032,39553978,2180-07-23 14:13:00,220277,98,98.0
5,10000032,39553978,2180-07-23 14:30:00,220045,93,93.0
6,10000032,39553978,2180-07-23 14:30:00,220179,95,95.0
7,10000032,39553978,2180-07-23 14:30:00,220180,59,59.0
8,10000032,39553978,2180-07-23 14:30:00,220181,67,67.0
9,10000032,39553978,2180-07-23 14:30:00,220277,97,97.0


In [14]:
result["vitals"] = result["itemid"].map(vitals_rev_dict)

result

,subject_id,stay_id,charttime,itemid,value,valuenum,vitals
0,10000032,39553978,2180-07-23 14:11:00,220179,84,84.0,NBPs
1,10000032,39553978,2180-07-23 14:11:00,220180,48,48.0,NBPd
2,10000032,39553978,2180-07-23 14:11:00,220181,56,56.0,NBPm
3,10000032,39553978,2180-07-23 14:12:00,220045,91,91.0,heartrate
4,10000032,39553978,2180-07-23 14:13:00,220277,98,98.0,spO2
5,10000032,39553978,2180-07-23 14:30:00,220045,93,93.0,heartrate
6,10000032,39553978,2180-07-23 14:30:00,220179,95,95.0,NBPs
7,10000032,39553978,2180-07-23 14:30:00,220180,59,59.0,NBPd
8,10000032,39553978,2180-07-23 14:30:00,220181,67,67.0,NBPm
9,10000032,39553978,2180-07-23 14:30:00,220277,97,97.0,spO2


group the chartevent work by itemids instead and then map the respective vitals

In [15]:
# queryforcount=f"""
# SELECT itemid, COUNT(*) AS n
# FROM read_csv_auto('{chartevents_path}')
# WHERE itemid IN {ids}
# GROUP BY itemid
# ORDER BY itemid
# """

# counts=con.execute(queryforcount).fetchdf()
# counts

In [16]:
# counts["vital"]=counts["itemid"].map(vitals_rev_dict)
# counts

thats the count of each vital present in the data frame so heartrate is obviously to be taken 

In [17]:
# queryfordups=f"""
# SELECT subject_id,stay_id,charttime,itemid,COUNT(*) as n
# FROM read_csv_auto('{chartevents_path}')
# WHERE itemid IN {ids}
# GROUP BY subject_id, stay_id, charttime, itemid
# HAVING COUNT(*)>1
# ORDER BY n DESC
# LIMIT 20
# """

# table1=con.execute(queryfordups).fetch_df()
# table1

In [18]:
# queryforvitalspecific = f""" 
# SELECT itemid, COUNT(*) AS totalrows, COUNT(*) FILTER(WHERE valuenum IS NULL) AS totalnull
# FROM read_csv_auto('{chartevents_path}')
# WHERE itemid in {ids}
# GROUP BY itemid
# ORDER BY totalrows DESC
# """

# table3 = con.execute(queryforvitalspecific).fetchdf()
# table3

In [19]:
# table3['vitals']=table3['itemid'].map(vitals_rev_dict)
# table3

In [20]:
# WERE GETTING THE CHARTEVENTS DF FOR ONCE NOW TO CHECK ITS STATS AND THEN WE SHALL PIVOT IT

In [21]:
querychev=f"""
SELECT subject_id,stay_id,charttime,itemid,valuenum
FROM read_csv_auto('{chartevents_path}')
WHERE itemid in {ids}
"""
rawchev=con.execute(querychev).fetch_df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [22]:
rawchev.head()

,subject_id,stay_id,charttime,itemid,valuenum
0,10000032,39553978,2180-07-23 14:11:00,220179,84.0
1,10000032,39553978,2180-07-23 14:11:00,220180,48.0
2,10000032,39553978,2180-07-23 14:11:00,220181,56.0
3,10000032,39553978,2180-07-23 14:12:00,220045,91.0
4,10000032,39553978,2180-07-23 14:13:00,220277,98.0


In [23]:
rawchev.shape

(35462414, 5)

In [24]:
rawchev["vitals"]=rawchev["itemid"].map(vitals_rev_dict)
rawchev.head()

,subject_id,stay_id,charttime,itemid,valuenum,vitals
0,10000032,39553978,2180-07-23 14:11:00,220179,84.0,NBPs
1,10000032,39553978,2180-07-23 14:11:00,220180,48.0,NBPd
2,10000032,39553978,2180-07-23 14:11:00,220181,56.0,NBPm
3,10000032,39553978,2180-07-23 14:12:00,220045,91.0,heartrate
4,10000032,39553978,2180-07-23 14:13:00,220277,98.0,spO2


In [25]:
pivoted = rawchev.pivot(values="valuenum",columns="vitals",index=["subject_id","stay_id","charttime"])
#df.pivot_table(values='v', index='a', columns='b', aggfunc='mean')

In [26]:
pivoted.head()

vitals                                   ARTdia  ARTmean  ARTsys  NBPd  NBPm  \
subject_id stay_id  charttime                                                  
10000032   39553978 2180-07-23 14:11:00     NaN      NaN     NaN  48.0  56.0   
                    2180-07-23 14:12:00     NaN      NaN     NaN   NaN   NaN   
                    2180-07-23 14:13:00     NaN      NaN     NaN   NaN   NaN   
                    2180-07-23 14:30:00     NaN      NaN     NaN  59.0  67.0   
                    2180-07-23 15:00:00     NaN      NaN     NaN  56.0  64.0   

vitals                                   NBPs  RR  heartrate  spO2  temp(C)  
subject_id stay_id  charttime                                                
10000032   39553978 2180-07-23 14:11:00  84.0 NaN        NaN   NaN      NaN  
                    2180-07-23 14:12:00   NaN NaN       91.0   NaN      NaN  
                    2180-07-23 14:13:00   NaN NaN        NaN  98.0      NaN  
                    2180-07-23 14:30:00  95.0 NaN       93.0  97.0      NaN  
                    2180-07-23 15:00:00  88.0 NaN       94.0  97.0      NaN

In [27]:
pivoted.isna().sum()


vitals
ARTdia       11968860
ARTmean      11966409
ARTsys       11968771
NBPd          6979062
NBPm          6983829
NBPs          6978011
RR           11903828
heartrate     3604682
spO2          3789736
temp(C)      11961908
dtype: int64

In [28]:
pivoted.isna().mean()*100

vitals
ARTdia       96.860898
ARTmean      96.841063
ARTsys       96.860178
NBPd         56.479749
NBPm         56.518328
NBPs         56.471244
RR           96.334611
heartrate    29.171762
spO2         30.669356
temp(C)      96.804637
dtype: float64

In [29]:
icustays.head()

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252
2,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535
3,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032
4,10001217,27703517,34592300,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-12-19 15:42:24,2157-12-20 14:27:41,0.948113


In [30]:
icustays.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit',
       'intime', 'outtime', 'los'],
      dtype='str')

In [31]:
icustays['intime']=pd.to_datetime(icustays['intime'])
icustays['outtime']=pd.to_datetime(icustays['outtime'])
icustays["stayhours"]=(icustays['outtime']-icustays['intime']).dt.total_seconds()/3600
icustays.head()
icustays["stayhours"].describe()

count    94444.000000
mean        87.120596
std        129.659365
min          0.030000
25%         26.309097
50%         47.175556
75%         92.701806
max       5433.673889
Name: stayhours, dtype: float64

In [32]:
icustayswmorethan4 = icustays[icustays["stayhours"] >= 4].copy()
icustayswmorethan4.head()

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,stayhours
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266,9.846389
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252,93.438056
2,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535,11.940833
3,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032,26.832778
4,10001217,27703517,34592300,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-12-19 15:42:24,2157-12-20 14:27:41,0.948113,22.754722


In [33]:
icustayswmorethan4.describe()

,subject_id,hadm_id,stay_id,intime,outtime,los,stayhours
count,9.373000e+04,9.373000e+04,9.373000e+04,93730,93730,93730.000000,93730.000000
mean,1.500421e+07,2.498339e+07,3.499724e+07,2153-10-26 00:51:53.396363,2153-10-29 16:37:56.498602,3.656980,87.767528
min,1.000003e+07,2.000009e+07,3.000015e+07,2110-01-11 10:16:06,2110-01-12 17:17:47,0.167095,4.010278
25%,1.251681e+07,2.248571e+07,3.250626e+07,2133-11-22 11:13:52.250000,2133-11-25 23:51:13.500000,1.108180,26.596319
50%,1.500336e+07,2.498502e+07,3.499794e+07,2153-09-30 08:03:48,2153-10-02 22:22:54,1.978628,47.487083
75%,1.751767e+07,2.746640e+07,3.748965e+07,2173-11-23 08:39:30,2173-11-27 14:39:08.250000,3.885909,93.261806
max,1.999999e+07,2.999983e+07,3.999986e+07,2214-07-22 17:05:53,2214-07-26 17:13:57,226.403079,5433.673889
std,2.884018e+06,2.883499e+06,2.886190e+06,NaN,NaN,5.414142,129.939397


In [34]:
stayids=icustayswmorethan4["stay_id"]
len(stayids)
# this is the list of stay ids which have more than 4 hrs of stay time so well filter jjust this form the entire dataset

93730

In [35]:
pivoted.columns
pivoted = pivoted.reset_index()
pivoted.columns

Index(['subject_id', 'stay_id', 'charttime', 'ARTdia', 'ARTmean', 'ARTsys',
       'NBPd', 'NBPm', 'NBPs', 'RR', 'heartrate', 'spO2', 'temp(C)'],
      dtype='str', name='vitals')

In [36]:
pivoted=pivoted[pivoted["stay_id"].isin(stayids)].copy()
pivoted.head()

vitals,subject_id,stay_id,charttime,ARTdia,ARTmean,ARTsys,NBPd,NBPm,NBPs,RR,heartrate,spO2,temp(C)
0,10000032,39553978,2180-07-23 14:11:00,NaN,NaN,NaN,48.0,56.0,84.0,NaN,NaN,NaN,NaN
1,10000032,39553978,2180-07-23 14:12:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,91.0,NaN,NaN
2,10000032,39553978,2180-07-23 14:13:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,98.0,NaN
3,10000032,39553978,2180-07-23 14:30:00,NaN,NaN,NaN,59.0,67.0,95.0,NaN,93.0,97.0,NaN
4,10000032,39553978,2180-07-23 15:00:00,NaN,NaN,NaN,56.0,64.0,88.0,NaN,94.0,97.0,NaN


In [37]:
print("Rows:", len(pivoted))
print("Unique ICU stays:", pivoted["stay_id"].nunique())
print("Expected ICU stays:", len(stayids))

Rows: 12348315
Unique ICU stays: 93729
Expected ICU stays: 93730


In [38]:
df=pivoted.sort_values(["stay_id","charttime"]).reset_index(drop=True)
df.head()

vitals,subject_id,stay_id,charttime,ARTdia,ARTmean,ARTsys,NBPd,NBPm,NBPs,RR,heartrate,spO2,temp(C)
0,12466550,30000153,2174-09-29 12:01:00,NaN,NaN,NaN,NaN,NaN,NaN,14.0,NaN,NaN,NaN
1,12466550,30000153,2174-09-29 12:05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN
2,12466550,30000153,2174-09-29 12:06:00,NaN,NaN,NaN,74.0,89.0,136.0,NaN,100.0,NaN,NaN
3,12466550,30000153,2174-09-29 13:00:00,NaN,NaN,NaN,77.0,84.0,113.0,NaN,104.0,100.0,NaN
4,12466550,30000153,2174-09-29 14:48:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,83.0,100.0,NaN


In [39]:
exid=df["stay_id"].iloc[0]
examplestay=df[df["stay_id"]==exid].copy()
examplestay.head()


vitals,subject_id,stay_id,charttime,ARTdia,ARTmean,ARTsys,NBPd,NBPm,NBPs,RR,heartrate,spO2,temp(C)
0,12466550,30000153,2174-09-29 12:01:00,NaN,NaN,NaN,NaN,NaN,NaN,14.0,NaN,NaN,NaN
1,12466550,30000153,2174-09-29 12:05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN
2,12466550,30000153,2174-09-29 12:06:00,NaN,NaN,NaN,74.0,89.0,136.0,NaN,100.0,NaN,NaN
3,12466550,30000153,2174-09-29 13:00:00,NaN,NaN,NaN,77.0,84.0,113.0,NaN,104.0,100.0,NaN
4,12466550,30000153,2174-09-29 14:48:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,83.0,100.0,NaN


In [40]:
examplestay["timegap"]=examplestay["charttime"].diff()
examplestay.head()

vitals,subject_id,stay_id,charttime,ARTdia,ARTmean,ARTsys,NBPd,NBPm,NBPs,RR,heartrate,spO2,temp(C),timegap
0,12466550,30000153,2174-09-29 12:01:00,NaN,NaN,NaN,NaN,NaN,NaN,14.0,NaN,NaN,NaN,NaT
1,12466550,30000153,2174-09-29 12:05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN,0 days 00:04:00
2,12466550,30000153,2174-09-29 12:06:00,NaN,NaN,NaN,74.0,89.0,136.0,NaN,100.0,NaN,NaN,0 days 00:01:00
3,12466550,30000153,2174-09-29 13:00:00,NaN,NaN,NaN,77.0,84.0,113.0,NaN,104.0,100.0,NaN,0 days 00:54:00
4,12466550,30000153,2174-09-29 14:48:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,83.0,100.0,NaN,0 days 01:48:00


In [41]:
df["time_diff"]=df.groupby("stay_id")["charttime"].diff()
df.head()

vitals,subject_id,stay_id,charttime,ARTdia,ARTmean,ARTsys,NBPd,NBPm,NBPs,RR,heartrate,spO2,temp(C),time_diff
0,12466550,30000153,2174-09-29 12:01:00,NaN,NaN,NaN,NaN,NaN,NaN,14.0,NaN,NaN,NaN,NaT
1,12466550,30000153,2174-09-29 12:05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN,0 days 00:04:00
2,12466550,30000153,2174-09-29 12:06:00,NaN,NaN,NaN,74.0,89.0,136.0,NaN,100.0,NaN,NaN,0 days 00:01:00
3,12466550,30000153,2174-09-29 13:00:00,NaN,NaN,NaN,77.0,84.0,113.0,NaN,104.0,100.0,NaN,0 days 00:54:00
4,12466550,30000153,2174-09-29 14:48:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,83.0,100.0,NaN,0 days 01:48:00


In [42]:
df["time_diff"].describe()

count                  12254586
mean     0 days 00:39:42.352446
std      0 days 03:44:12.100981
min             0 days 00:01:00
25%             0 days 00:05:00
50%             0 days 00:58:00
75%             0 days 01:00:00
max           365 days 00:04:00
Name: time_diff, dtype: object

In [43]:
df.loc[df["time_diff"].nlargest(10).index,
    ["subject_id", "stay_id", "charttime", "time_diff"]
]

vitals,subject_id,stay_id,charttime,time_diff
5965964,18138079,34773087,2181-01-02 23:39:00,365 days 00:04:00
750223,16661755,30586760,2175-06-11 20:00:00,364 days 14:59:00
8120491,10253349,36520389,2190-03-11 14:02:00,80 days 23:02:00
5147891,16186431,34123444,2140-10-12 03:26:00,32 days 06:25:00
10523331,13381135,38494776,2152-03-24 15:24:00,30 days 15:24:00
9852206,19611909,37946973,2160-11-20 23:26:00,30 days 00:01:00
11752872,18744840,39510663,2114-11-08 13:00:00,25 days 17:00:00
373675,19157548,30290852,2128-06-11 18:00:00,24 days 19:58:00
7253022,15281657,35838821,2128-02-05 03:09:00,23 days 18:08:00
2112317,13333927,31699045,2139-01-19 10:00:00,23 days 11:10:00


we see the same 2 subject ids having an exorbitant gap

In [44]:
vital_cols=[
    "ARTdia", "ARTmean", "ARTsys",
    "NBPd", "NBPm", "NBPs",
    "RR", "heartrate", "spO2", "temp(C)"
]

In [45]:
df_check=df.merge(icustays[["stay_id","intime","outtime"]], on="stay_id",how='left')
#pd.merge(df1, df2, on='key', how='inner') 
df_check.head()

,subject_id,stay_id,charttime,ARTdia,ARTmean,ARTsys,NBPd,NBPm,NBPs,RR,heartrate,spO2,temp(C),time_diff,intime,outtime
0,12466550,30000153,2174-09-29 12:01:00,NaN,NaN,NaN,NaN,NaN,NaN,14.0,NaN,NaN,NaN,NaT,2174-09-29 12:09:00,2174-10-01 03:26:10
1,12466550,30000153,2174-09-29 12:05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN,0 days 00:04:00,2174-09-29 12:09:00,2174-10-01 03:26:10
2,12466550,30000153,2174-09-29 12:06:00,NaN,NaN,NaN,74.0,89.0,136.0,NaN,100.0,NaN,NaN,0 days 00:01:00,2174-09-29 12:09:00,2174-10-01 03:26:10
3,12466550,30000153,2174-09-29 13:00:00,NaN,NaN,NaN,77.0,84.0,113.0,NaN,104.0,100.0,NaN,0 days 00:54:00,2174-09-29 12:09:00,2174-10-01 03:26:10
4,12466550,30000153,2174-09-29 14:48:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,83.0,100.0,NaN,0 days 01:48:00,2174-09-29 12:09:00,2174-10-01 03:26:10


In [46]:
timethres=pd.Timedelta(hours=60)
diffthres=pd.Timedelta(days=1)
df_timeclean=df_check[
    (df_check["charttime"]>=df_check["intime"]-timethres)
      & 
    (df_check["charttime"]<=df_check["outtime"]+timethres)
     & 
    ((df_check["time_diff"].isna()))
      | 
    (df_check["time_diff"]<=diffthres)
    ].copy()

In [47]:
uniform=(
    df_timeclean.set_index("charttime")
      .groupby("stay_id")[vital_cols]
      .resample("20min")
      .mean()
      .reset_index()
)

In [48]:
uniform.head()

,stay_id,charttime,ARTdia,ARTmean,ARTsys,NBPd,NBPm,NBPs,RR,heartrate,spO2,temp(C)
0,30000153,2174-09-29 12:00:00,NaN,NaN,NaN,74.0,89.0,136.0,14.0,100.0,100.0,NaN
1,30000153,2174-09-29 12:20:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,30000153,2174-09-29 12:40:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,30000153,2174-09-29 13:00:00,NaN,NaN,NaN,77.0,84.0,113.0,NaN,104.0,100.0,NaN
4,30000153,2174-09-29 13:20:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [49]:
uniform=uniform.sort_values(["stay_id", "charttime"])
uniform=uniform.set_index("charttime")

In [50]:
uniform.head()

,stay_id,ARTdia,ARTmean,ARTsys,NBPd,NBPm,NBPs,RR,heartrate,spO2,temp(C)
charttime,,,,,,,,,,,
2174-09-29 12:00:00,30000153,NaN,NaN,NaN,74.0,89.0,136.0,14.0,100.0,100.0,NaN
2174-09-29 12:20:00,30000153,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2174-09-29 12:40:00,30000153,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2174-09-29 13:00:00,30000153,NaN,NaN,NaN,77.0,84.0,113.0,NaN,104.0,100.0,NaN
2174-09-29 13:20:00,30000153,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [51]:
uniform.groupby("stay_id").size().describe()

count    93729.000000
mean       258.874948
std        388.894853
min          1.000000
25%         74.000000
50%        138.000000
75%        277.000000
max      16297.000000
dtype: float64

In [52]:
uniform.groupby("stay_id").size().sort_values(ascending=False).head(10)

stay_id
36032605    16297
36307509    11493
39510663    10078
30359303     9790
31492392     9159
35629939     8731
39245279     8065
32380519     7439
38018615     7315
31879957     7169
dtype: int64

we get an abnormal reading spanning 1 year so it got split into 106830 samples of 5 minutes

In [53]:
uniform = uniform.reset_index()

uniform["time_diff"] = (
    uniform.groupby("stay_id")["charttime"].diff()
)

uniform = uniform.set_index("charttime")

In [54]:
uniform["time_diff"].describe()

count           24170361
mean     0 days 00:20:00
std      0 days 00:00:00
min      0 days 00:20:00
25%      0 days 00:20:00
50%      0 days 00:20:00
75%      0 days 00:20:00
max      0 days 00:20:00
Name: time_diff, dtype: object

FINALLY all of the data has been made into workable time series data of 5 mins intervals after removing the stray huge interval samples

In [55]:
uniform = uniform.drop(columns=["time_diff"])
(uniform.isna().mean() * 100).sort_values(ascending=False)

ARTdia       98.526901
ARTsys       98.526596
ARTmean      98.518733
temp(C)      98.464937
RR           98.185763
NBPm         78.447393
NBPd         78.433636
NBPs         78.429799
spO2         66.355087
heartrate    65.767428
stay_id       0.000000
dtype: float64

In [56]:
uniform[vitals].notna().mean().sort_values(ascending=False)

heartrate    0.342326
spO2         0.336449
NBPs         0.215702
NBPd         0.215664
NBPm         0.215526
RR           0.018142
temp(C)      0.015351
ARTmean      0.014813
ARTsys       0.014734
ARTdia       0.014731
dtype: float64

In [57]:
for i in vitals:
    actual = uniform[uniform[i].notna()].copy()

    actual["gap"] = (actual.index.to_series().groupby(actual["stay_id"]).diff())
    print(f"\n{i}")
    print(actual["gap"].dropna().describe())


spO2
count                   8070002
mean     0 days 00:59:26.438967
std      0 days 00:47:09.164866
min             0 days 00:20:00
25%             0 days 01:00:00
50%             0 days 01:00:00
75%             0 days 01:00:00
max            25 days 17:00:00
Name: gap, dtype: object

temp(C)
count                    362611
mean     0 days 01:23:45.206626
std      0 days 08:01:53.986703
min             0 days 00:20:00
25%             0 days 01:00:00
50%             0 days 01:00:00
75%             0 days 01:00:00
max            46 days 02:40:00
Name: gap, dtype: object

heartrate
count                   8212494
mean     0 days 00:58:37.565163
std      0 days 00:39:45.424786
min             0 days 00:20:00
25%             0 days 01:00:00
50%             0 days 01:00:00
75%             0 days 01:00:00
max            24 days 21:00:00
Name: gap, dtype: object

ARTsys
count                    350532
mean     0 days 01:06:24.896100
std      0 days 06:56:25.747798
min             0 days 00:2

so for: spo2: avg is 57 mins, temp is 1hr 19 mins, heart rate is 56mins art sys and dia (and hence mean) is 1 hr 2 mins nbps nbpd and nbpm is 1hr 21 mins rr is 55 mins so take the avrage around 1.5 hrs

In [58]:
uniform.shape

(24264090, 11)

In [59]:
uniform.memory_usage(deep=True).sum() / 1024**3

np.float64(2.169378697872162)

In [60]:
for col in vital_cols:
    uniform[col] = uniform.groupby("stay_id")[col].ffill(limit=4)

In [61]:
uniform[vital_cols].isna().sum()

ARTdia       23262017
ARTmean      23256853
ARTsys       23261819
NBPd          8644840
NBPm          8659586
NBPs          8642934
RR           22170018
heartrate      571545
spO2           865676
temp(C)      23125160
dtype: int64

In [62]:
uniform[vital_cols].isna().sum()

ARTdia       23262017
ARTmean      23256853
ARTsys       23261819
NBPd          8644840
NBPm          8659586
NBPs          8642934
RR           22170018
heartrate      571545
spO2           865676
temp(C)      23125160
dtype: int64

In [63]:
uniform[["heartrate", "spO2"]].notna().all(axis=1).sum()

np.int64(23357617)

In [64]:
(uniform[vital_cols].isna().mean() * 100).sort_values(ascending=False)

ARTdia       95.870140
ARTsys       95.869324
ARTmean      95.848857
temp(C)      95.306109
RR           91.369666
NBPm         35.688897
NBPd         35.628124
NBPs         35.620268
spO2          3.567725
heartrate     2.355518
dtype: float64

In [65]:
uniform = uniform.drop(columns=["ARTdia","ARTsys"])
uniform.head()

,stay_id,ARTmean,NBPd,NBPm,NBPs,RR,heartrate,spO2,temp(C)
charttime,,,,,,,,,
2174-09-29 12:00:00,30000153,NaN,74.0,89.0,136.0,14.0,100.0,100.0,NaN
2174-09-29 12:20:00,30000153,NaN,74.0,89.0,136.0,14.0,100.0,100.0,NaN
2174-09-29 12:40:00,30000153,NaN,74.0,89.0,136.0,14.0,100.0,100.0,NaN
2174-09-29 13:00:00,30000153,NaN,77.0,84.0,113.0,14.0,104.0,100.0,NaN
2174-09-29 13:20:00,30000153,NaN,77.0,84.0,113.0,14.0,104.0,100.0,NaN


In [66]:
# d_items[d_items["label"].str.contains(
#     "glucose|fio2|oxygen|gcs|pain|urine|respiratory|ventilator",
#     case=False,
#     na=False
# )][["itemid", "label"]].head(20)

In [67]:
# candidate_ids = (
#     # 
#     220621,   # Glucose serum
#     220739,   # GCS Eye Opening
#     220210,   # Respiratory Rate
# )

In [68]:
uniform.sort_values(['stay_id','charttime'])
uniform.head(10)
#df.sort_values('a', ascending=False)

,stay_id,ARTmean,NBPd,NBPm,NBPs,RR,heartrate,spO2,temp(C)
charttime,,,,,,,,,
2174-09-29 12:00:00,30000153,NaN,74.0,89.0,136.0,14.0,100.0,100.0,NaN
2174-09-29 12:20:00,30000153,NaN,74.0,89.0,136.0,14.0,100.0,100.0,NaN
2174-09-29 12:40:00,30000153,NaN,74.0,89.0,136.0,14.0,100.0,100.0,NaN
2174-09-29 13:00:00,30000153,NaN,77.0,84.0,113.0,14.0,104.0,100.0,NaN
2174-09-29 13:20:00,30000153,NaN,77.0,84.0,113.0,14.0,104.0,100.0,NaN
2174-09-29 13:40:00,30000153,NaN,77.0,84.0,113.0,NaN,104.0,100.0,NaN
2174-09-29 14:00:00,30000153,NaN,77.0,84.0,113.0,NaN,104.0,100.0,NaN
2174-09-29 14:20:00,30000153,NaN,77.0,84.0,113.0,NaN,104.0,100.0,NaN
2174-09-29 14:40:00,30000153,NaN,NaN,NaN,NaN,NaN,83.0,100.0,NaN


In [69]:
# uniform["stay_id"].unique()[:10]

In [71]:
import pandas as pd
import pyarrow as pa

print(pd.__version__)
print(pa.__version__)

3.0.5
25.0.1


In [72]:
import pyarrow
uniform.to_parquet("uniform_20min.parquet")


Parquet is basically a data file format designed for big datasets. 
Think of it as a much smarter alternative to CSV.

In [73]:
import os
os.path.getsize("uniform_20min.parquet") / 1024**3

0.22740452364087105

# THIS IS THE CHECKPOINT FOR RECOVERING THE UNIFORM DATAFRAME WHICH WAS SAVED AS A PARQUET. RUN THIS LINE FOR IMPLMENTATIONS HENCE FORTH

In [74]:
uniform = pd.read_parquet("uniform_20min.parquet")

In [77]:
print(uniform.index.name)
print(uniform.columns)

charttime
Index(['stay_id', 'ARTmean', 'NBPd', 'NBPm', 'NBPs', 'RR', 'heartrate', 'spO2',
       'temp(C)'],
      dtype='str')


In [86]:
diffs = (
    uniform.groupby("stay_id", sort=False)
    .apply(lambda x: x.index.to_series().diff())
    .dropna()
)
#check the spacing thruout
diffs.value_counts().head()

charttime
0 days 00:20:00    24170361
Name: count, dtype: int64

In [81]:
uniform = uniform.sort_values(["stay_id", uniform.index.name])

In [88]:
#taking 2 hr windows now
window_size = 6
uniform["window_id"]=(uniform.groupby("stay_id")).cumcount()//window_size
uniform.head(10)


,stay_id,ARTmean,NBPd,NBPm,NBPs,RR,heartrate,spO2,temp(C),window_id
charttime,,,,,,,,,,
2174-09-29 12:00:00,30000153,NaN,74.0,89.0,136.0,14.0,100.0,100.0,NaN,0
2174-09-29 12:20:00,30000153,NaN,74.0,89.0,136.0,14.0,100.0,100.0,NaN,0
2174-09-29 12:40:00,30000153,NaN,74.0,89.0,136.0,14.0,100.0,100.0,NaN,0
2174-09-29 13:00:00,30000153,NaN,77.0,84.0,113.0,14.0,104.0,100.0,NaN,0
2174-09-29 13:20:00,30000153,NaN,77.0,84.0,113.0,14.0,104.0,100.0,NaN,0
2174-09-29 13:40:00,30000153,NaN,77.0,84.0,113.0,NaN,104.0,100.0,NaN,0
2174-09-29 14:00:00,30000153,NaN,77.0,84.0,113.0,NaN,104.0,100.0,NaN,1
2174-09-29 14:20:00,30000153,NaN,77.0,84.0,113.0,NaN,104.0,100.0,NaN,1
2174-09-29 14:40:00,30000153,NaN,NaN,NaN,NaN,NaN,83.0,100.0,NaN,1


In [90]:
vitalsnew=[
    "ARTmean",
    "NBPd", "NBPm", "NBPs",
    "RR", "heartrate", "spO2", "temp(C)"
]

In [93]:
completeness = (
    uniform[vitalsnew].notna()
    .groupby([uniform["stay_id"], uniform["window_id"]])
    .mean()
)

In [94]:
completeness.describe()

,ARTmean,NBPd,NBPm,NBPs,RR,heartrate,spO2,temp(C)
count,4.085640e+06,4.085640e+06,4.085640e+06,4.085640e+06,4.085640e+06,4.085640e+06,4.085640e+06,4.085640e+06
mean,4.121595e-02,6.458447e-01,6.451924e-01,6.459248e-01,8.558833e-02,9.764311e-01,9.639701e-01,4.656437e-02
std,1.970919e-01,4.564158e-01,4.568755e-01,4.563913e-01,2.233216e-01,1.241950e-01,1.566492e-01,2.033344e-01
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00
50%,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00
75%,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00
max,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
